In [5]:
import h3
import json
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt


crowd_mapping_df = pd.read_csv("../data/raw/popular_times/popular_times_malang.csv")
poi_grid_df = pd.read_csv("../data/processed/poi_grid_malang.csv")

venues_coord_df = crowd_mapping_df.copy()
venues_coord_df = venues_coord_df[["venue_name", "lat", "lon", "zone"]]
venues_coord_df.drop_duplicates(subset=["venue_name"], inplace=True)

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)  
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1 
    dlon = lon2 - lon1
    
    a = (np.sin(dlat / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)

    return R * 2 * np.arcsin(np.sqrt(a))

def tier_define(distance):
    if distance < 200:
        return 1
    elif distance <= 500:
        return 2
    elif distance <= 1000:
        return 3
    else:
        return 4




In [6]:
rows = []
MAX_DISTANCE = 1000

for _, grid in poi_grid_df.iterrows():

    grid_lat = grid.get("lat")
    grid_lon = grid.get("lon")

    venues_coord_df["meter_distance"] = haversine_vectorized(
        grid_lat, grid_lon,
        venues_coord_df["lat"],
        venues_coord_df["lon"]
    )

    in_radius = venues_coord_df[
        venues_coord_df["meter_distance"] <= MAX_DISTANCE
    ].copy()

    if len(in_radius) == 0:
        rows.append({
            "grid_id"        : grid.get("grid_id"),
            "grid_lat"       : grid_lat,
            "grid_lon"       : grid_lon,
            "venue_name"     : None,
            "venue_lat"      : None,
            "venue_lon"      : None,
            "meter_distance" : None,
            "zone"           : None,
            "tier"           : 4,
            "is_nearest"     : False,
        })
        continue

    nearest_idx = in_radius["meter_distance"].idxmin()

    for idx, venue in in_radius.iterrows(): 
        rows.append({
            "grid_id" : grid.get("grid_id"),
            "grid_lat" : grid_lat,
            "grid_lon" : grid_lon,
            "venue_name"  : venue["venue_name"],
            "venue_lat" : venue["lat"],
            "venue_lon" : venue["lon"],
            "meter_distance" : round(venue["meter_distance"], 2),
            "zone" : venue["zone"],
            "tier" :  tier_define(venue["meter_distance"]),
            "is_nearest" :  idx == nearest_idx
        })

df_grid_mapping = pd.DataFrame(rows)
df_grid_mapping.to_csv("../data/processed/venue_to_grid_mapping.csv", index=False)



    
